# Leak-Free Drug Prediction Evaluation

Compare predictions from the clean pipeline against ground truth.

Metrics:
- Top-1 accuracy: Is the #1 predicted drug in the ground truth?
- Top-3 accuracy: Is any of the top-3 predicted drugs in the ground truth?
- Per-drug precision/recall
- Comparison with old (leaky) pipeline results

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

_HERE = os.path.dirname(os.path.abspath('__file__'))

# Paths
ground_truth_path = os.path.join(_HERE, 'ground_truth.json')
old_ground_truth_path = os.path.join(_HERE, 'drug_ground_truth.json')

# Clean pipeline predictions (update after running predict_drugs_clean.py)
clean_pred_dir = os.path.join(_HERE, 'drug', 'clean')

# Old pipeline predictions (for comparison)
old_pred_dir = os.path.join(_HERE, 'drug', 'all_drugs')

## 1. Load Ground Truth

In [ ]:
with open(ground_truth_path, 'r') as f:
    ground_truth = json.load(f)

print(f'Loaded ground truth for {len(ground_truth)} patients')

# Summary stats
for visit in ['Visit_1', 'Visit_2', 'Visit_3']:
    prescribed_counts = Counter()
    n_with_plan = 0
    n_reconciled = 0
    for pid, visits in ground_truth.items():
        vdata = visits.get(visit, {})
        drugs = vdata.get('prescribed_drugs', [])
        if drugs:
            n_with_plan += 1
        for d in drugs:
            prescribed_counts[d] += 1
        if vdata.get('source') == 'reconciled_from_later_observation':
            n_reconciled += 1
    print(f'\n{visit}:')
    print(f'  Patients with prescribed drugs: {n_with_plan}')
    print(f'  Reconciled from later observation: {n_reconciled}')
    print(f'  Drug distribution: {dict(prescribed_counts.most_common())}')

## 2. Compare New vs Old Ground Truth

In [ ]:
if os.path.exists(old_ground_truth_path):
    with open(old_ground_truth_path, 'r') as f:
        old_gt = json.load(f)
    
    print(f'Old ground truth: {len(old_gt)} patients')
    print(f'New ground truth: {len(ground_truth)} patients')
    
    # Compare per-visit drug sets
    for visit in ['Visit_1', 'Visit_2', 'Visit_3']:
        matches = 0
        mismatches = 0
        mismatch_examples = []
        common_pids = set(ground_truth.keys()) & set(old_gt.keys())
        
        for pid in common_pids:
            new_drugs = set(ground_truth[pid].get(visit, {}).get('prescribed_drugs', []))
            old_drugs = set()
            old_visit = old_gt.get(pid, {}).get(visit, {})
            if 'drug_binary' in old_visit:
                old_drugs = {d for d, v in old_visit['drug_binary'].items() if v == 1}
            elif 'active_drugs' in old_visit:
                old_drugs = set(old_visit['active_drugs'])
            
            if new_drugs == old_drugs:
                matches += 1
            else:
                mismatches += 1
                if len(mismatch_examples) < 5:
                    mismatch_examples.append({
                        'pid': pid,
                        'new': sorted(new_drugs),
                        'old': sorted(old_drugs),
                        'new_only': sorted(new_drugs - old_drugs),
                        'old_only': sorted(old_drugs - new_drugs),
                    })
        
        print(f'\n{visit}: {matches} matches, {mismatches} mismatches out of {len(common_pids)}')
        for ex in mismatch_examples:
            print(f'  {ex["pid"]}: new={ex["new"]}, old={ex["old"]}')
            if ex['new_only']:
                print(f'    New only: {ex["new_only"]}')
            if ex['old_only']:
                print(f'    Old only: {ex["old_only"]}')
else:
    print('Old ground truth not found, skipping comparison')

## 3. Load and Evaluate Predictions

In [ ]:
def load_predictions(pred_dir, visit_num, pattern='clean'):
    """Load prediction CSV files matching the pattern."""
    results = {}
    for fname in os.listdir(pred_dir):
        if fname.endswith('.csv') and f'v{visit_num}' in fname and pattern in fname:
            df = pd.read_csv(os.path.join(pred_dir, fname))
            model_name = fname.replace('.csv', '')
            results[model_name] = df
            print(f'Loaded {fname}: {len(df)} patients')
    return results


def evaluate_predictions(pred_df, ground_truth, visit_num):
    """Compute accuracy metrics."""
    visit_name = f'Visit_{visit_num}'
    
    top1_correct = 0
    top3_any_correct = 0
    total = 0
    per_drug_tp = Counter()  # predicted and in GT
    per_drug_fp = Counter()  # predicted but not in GT
    per_drug_fn = Counter()  # in GT but not predicted
    
    for _, row in pred_df.iterrows():
        pid = row['patient_id']
        gt_data = ground_truth.get(pid, {}).get(visit_name, {})
        gt_drugs = set(gt_data.get('prescribed_drugs', []))
        
        if not gt_drugs:
            continue
        
        total += 1
        predicted = []
        for rank in [1, 2, 3]:
            drug = row.get(f'rank_{rank}')
            if pd.notna(drug) and drug:
                predicted.append(drug)
        
        # Top-1 accuracy
        if predicted and predicted[0] in gt_drugs:
            top1_correct += 1
        
        # Top-3 accuracy (any predicted drug in GT)
        if any(d in gt_drugs for d in predicted):
            top3_any_correct += 1
        
        # Per-drug metrics
        predicted_set = set(predicted)
        for d in predicted_set:
            if d in gt_drugs:
                per_drug_tp[d] += 1
            else:
                per_drug_fp[d] += 1
        for d in gt_drugs:
            if d not in predicted_set:
                per_drug_fn[d] += 1
    
    metrics = {
        'total': total,
        'top1_accuracy': top1_correct / total if total > 0 else 0,
        'top3_accuracy': top3_any_correct / total if total > 0 else 0,
        'top1_correct': top1_correct,
        'top3_correct': top3_any_correct,
    }
    
    # Per-drug precision and recall
    all_drugs = sorted(set(list(per_drug_tp) + list(per_drug_fp) + list(per_drug_fn)))
    drug_metrics = []
    for d in all_drugs:
        tp = per_drug_tp[d]
        fp = per_drug_fp[d]
        fn = per_drug_fn[d]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        drug_metrics.append({
            'drug': d, 'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall, 'f1': f1
        })
    
    return metrics, pd.DataFrame(drug_metrics)

In [ ]:
# Evaluate clean pipeline predictions
for visit_num in [1, 2, 3]:
    print(f'\n{"="*60}')
    print(f'Visit {visit_num} Evaluation')
    print(f'{"="*60}')
    
    preds = load_predictions(clean_pred_dir, visit_num, pattern='clean')
    
    if not preds:
        print(f'No predictions found for Visit {visit_num}')
        continue
    
    for model_name, pred_df in preds.items():
        metrics, drug_df = evaluate_predictions(pred_df, ground_truth, visit_num)
        
        print(f'\nModel: {model_name}')
        print(f'  Total evaluated: {metrics["total"]}')
        print(f'  Top-1 accuracy: {metrics["top1_accuracy"]:.3f} ({metrics["top1_correct"]}/{metrics["total"]})')
        print(f'  Top-3 accuracy: {metrics["top3_accuracy"]:.3f} ({metrics["top3_correct"]}/{metrics["total"]})')
        print(f'\n  Per-drug metrics:')
        print(drug_df.to_string(index=False))

## 4. Leakage Comparison (Clean vs Old Pipeline)

In [ ]:
# Compare clean vs old pipeline predictions
comparison_rows = []

for visit_num in [1, 2, 3]:
    # Clean pipeline
    clean_preds = load_predictions(clean_pred_dir, visit_num, pattern='clean')
    # Old pipeline
    old_preds = load_predictions(old_pred_dir, visit_num, pattern='top3')
    
    for model_name, pred_df in clean_preds.items():
        metrics, _ = evaluate_predictions(pred_df, ground_truth, visit_num)
        comparison_rows.append({
            'pipeline': 'clean',
            'visit': visit_num,
            'model': model_name,
            'top1_acc': metrics['top1_accuracy'],
            'top3_acc': metrics['top3_accuracy'],
            'n': metrics['total'],
        })
    
    # For old pipeline, use old ground truth if available
    if os.path.exists(old_ground_truth_path):
        with open(old_ground_truth_path, 'r') as f:
            old_gt_data = json.load(f)
        
        # Convert old GT format to match new format
        old_gt_converted = {}
        for pid, visits in old_gt_data.items():
            old_gt_converted[pid] = {}
            for vname, vdata in visits.items():
                if 'active_drugs' in vdata:
                    old_gt_converted[pid][vname] = {
                        'prescribed_drugs': vdata['active_drugs']
                    }
                elif 'drug_binary' in vdata:
                    old_gt_converted[pid][vname] = {
                        'prescribed_drugs': [d for d, v in vdata['drug_binary'].items() if v == 1]
                    }
        
        for model_name, pred_df in old_preds.items():
            metrics, _ = evaluate_predictions(pred_df, old_gt_converted, visit_num)
            comparison_rows.append({
                'pipeline': 'old (leaky)',
                'visit': visit_num,
                'model': model_name,
                'top1_acc': metrics['top1_accuracy'],
                'top3_acc': metrics['top3_accuracy'],
                'n': metrics['total'],
            })

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows)
    print('\nPipeline Comparison:')
    print(comp_df.to_string(index=False))
else:
    print('No predictions available for comparison yet')

## 5. Visualization

In [ ]:
# Plot per-drug accuracy for each visit
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, visit_num in enumerate([1, 2, 3]):
    ax = axes[i]
    preds = load_predictions(clean_pred_dir, visit_num, pattern='clean')
    
    if not preds:
        ax.set_title(f'Visit {visit_num} (no data)')
        continue
    
    # Use first model's results
    model_name, pred_df = next(iter(preds.items()))
    _, drug_df = evaluate_predictions(pred_df, ground_truth, visit_num)
    
    if not drug_df.empty:
        drug_df = drug_df.sort_values('f1', ascending=True)
        ax.barh(drug_df['drug'], drug_df['f1'], color='steelblue')
        ax.set_xlabel('F1 Score')
        ax.set_title(f'Visit {visit_num} - Per-Drug F1')
        ax.set_xlim(0, 1)
    else:
        ax.set_title(f'Visit {visit_num} (no data)')

plt.tight_layout()
plt.savefig(os.path.join(_HERE, 'drug', 'clean', 'per_drug_f1.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved per_drug_f1.png')

In [ ]:
# Summary table of top-1 and top-3 accuracy across visits
summary_rows = []

for visit_num in [1, 2, 3]:
    preds = load_predictions(clean_pred_dir, visit_num, pattern='clean')
    for model_name, pred_df in preds.items():
        metrics, _ = evaluate_predictions(pred_df, ground_truth, visit_num)
        summary_rows.append({
            'Visit': visit_num,
            'Model': model_name,
            'N': metrics['total'],
            'Top-1 Acc': f"{metrics['top1_accuracy']:.1%}",
            'Top-3 Acc': f"{metrics['top3_accuracy']:.1%}",
        })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    print('\nClean Pipeline Results Summary:')
    print(summary_df.to_string(index=False))
else:
    print('No results to summarize yet. Run predict_drugs_clean.py first.')